# DC-OPF Objective and Solve-Time Check: Pyomo/HiGHS vs PYPOWER on IEEE Cases

Companion to `acopf_objective_check.ipynb`. Does the DC-OPF formulation from [01_dcopf_pyomo.ipynb](https://github.com/vjkhtw/pjm5-opf/blob/main/notebooks/01_dcopf_pyomo.ipynb), solved with HiGHS in Pyomo, reach the same objective as PYPOWER's `dcopf` ([02_dcopf_pypower_reference.ipynb](https://github.com/vjkhtw/pjm5-opf/blob/main/notebooks/02_dcopf_pypower_reference.ipynb)), and how long does each take? Only the **total generation cost** and the **solve time** are compared, on the same six PGLib-OPF cases as the AC notebook (14, 30, 39 (EPRI), 57, 118 and 162 (DTC) buses).

**Contents**
* Setup
* Data pipeline: case file to solver input
* DC model: what changes for tap ratios and bus shunts
* Pyomo DC-OPF model
* Sanity check on the 5-bus case
* Objective comparison across IEEE cases
* Solve time

## Setup

In [1]:
# Dependencies are installed from requirements.txt before launching the notebook.


In [2]:
import re, time, urllib.request
import numpy as np, pandas as pd
import pyomo.environ as pyo
from pypower.api import dcopf, ppoption

## Data pipeline: case file to solver input
Same pipeline as the AC notebook: `pglib-opf .m file` → `read_matpower()` → `case` dict (`baseMVA` plus the MATPOWER-layout arrays `bus`, `gen`, `gencost`, `branch`) → `solve_pyomo_dcopf(case)` and `solve_pypower_dc(case)`. Both solvers read the same dictionary.

In [3]:
# Pinned to the commit behind tag v23.07 of power-grid-lib/pglib-opf (a commit hash cannot move; "master" can)
PGLIB_COMMIT = "dc6be4b2f85ca0e776952ec22cbd4c22396ea5a3"
BASE_URL = f"https://raw.githubusercontent.com/power-grid-lib/pglib-opf/{PGLIB_COMMIT}/"

def read_matpower(name):
    """Download a pglib-opf case (e.g. 'pglib_opf_case14_ieee.m') and return the MATPOWER-style dict."""
    txt = urllib.request.urlopen(BASE_URL + name).read().decode()
    txt = "\n".join(l.split("%")[0] for l in txt.splitlines())          # strip comments
    case = {"baseMVA": float(re.search(r"mpc\.baseMVA\s*=\s*([\d.]+)", txt).group(1))}
    for key in ["bus", "gen", "gencost", "branch"]:
        body = re.search(r"mpc\.%s\s*=\s*\[(.*?)\];" % key, txt, re.S).group(1)
        case[key] = np.array([r.split() for r in body.replace("\t", " ").split(";") if r.strip()], dtype=float)
    return case

CASES = ["14_ieee", "30_ieee", "39_epri", "57_ieee", "118_ieee", "162_ieee_dtc"]
cases = {n: read_matpower(f"pglib_opf_case{n}.m") for n in CASES}

## DC model: what changes for tap ratios and bus shunts
DC-OPF uses only part of the case data: bus demand, generator limits and linear costs, branch reactance, thermal limits and angle limits. Resistance, line charging, reactive demand, reactive limits and voltage bounds are not used. Two network features still matter for matching PYPOWER, and neither is an extra optimisation constraint:

* **Transformer tap ratio $\tau$** (0 in the file means $\tau = 1$). The branch flow is $P_{ij} = \dfrac{\theta_i - \theta_j}{x_{ij}\,\tau_{ij}}$, so the susceptance is $1/(x\tau)$ instead of $1/x$.
* **Bus shunt conductance $G_s$** (MW at 1.0 p.u. voltage). It is treated as extra demand, so the demand used at the bus is $P_d + G_s$. The shunt susceptance $B_s$ is a reactive quantity and is not used.

With $\tau = 1$ and $G_s = 0$ this reduces to the formulation in `01_dcopf_pyomo.ipynb`.

In [4]:
rows = []
for n, c in cases.items():
    br, bus = c["branch"], c["bus"]
    rows.append({"Case": n, "Buses": len(bus), "Generators": len(c["gen"]), "Branches": len(br),
                 "Tapped branches": int((br[:, 8] != 0).sum()),
                 "Buses with Gs > 0": int((bus[:, 4] != 0).sum()),
                 "Demand (MW)": bus[:, 2].sum()})
pd.DataFrame(rows).style.format({"Demand (MW)": "{:,.0f}"}).hide(axis="index")

Case,Buses,Generators,Branches,Tapped branches,Buses with Gs > 0,Demand (MW)
14_ieee,14,5,20,3,0,259
30_ieee,30,6,41,7,0,283
39_epri,39,10,46,11,0,"6,254"
57_ieee,57,7,80,17,0,"1,251"
118_ieee,118,54,186,11,0,"4,242"
162_ieee_dtc,162,12,284,91,0,"7,239"


## Pyomo DC-OPF model
Variables are $P_g$ for every generator and $\theta_i$ for every bus.

$$\min \; \sum_g c_g P_g$$

subject to generator limits, nodal balance $\sum P_g - (P_d + G_s) = \sum P_{ij}$ at every bus, line limits $-F^{max}_{ij} \le P_{ij} \le F^{max}_{ij}$ (the case's MVA rating used as a symmetric MW limit, as in `01`), branch angle-difference limits (±30°) and a reference angle $\theta_{ref} = 0$. The model is an LP, so HiGHS returns the global optimum.

The function returns the status, the cost, and two times: `t_build` (constructing the Pyomo model) and `t_solve` (the `solver.solve(m)` call, which includes passing the model to HiGHS, solving, and reading the result back).

In [5]:
def solve_pyomo_dcopf(case):
    t0 = time.perf_counter()
    base = case["baseMVA"]; bus, gen, gcost, br = case["bus"], case["gen"], case["gencost"], case["branch"]
    nb, ng, nl = len(bus), len(gen), len(br)
    idx = {int(b): k for k, b in enumerate(bus[:, 0])}
    ref = idx[int(bus[bus[:, 1] == 3, 0][0])]
    assert np.allclose(gcost[:, 4], 0), "only linear costs (c2 = 0) are handled"
    assert np.allclose(gcost[:, 6], 0), "constant cost term c0 is not in the objective"
    assert np.allclose(br[:, 9], 0), "phase-shifting transformers are not handled"
    assert (gen[:, 7] == 1).all(), "out-of-service generators (status = 0) are not handled"
    assert (br[:, 10] == 1).all(), "out-of-service branches (status = 0) are not handled"

    Pd = (bus[:, 2] + bus[:, 4]) / base                     # demand plus shunt conductance at 1 p.u.
    gbus = [idx[int(b)] for b in gen[:, 0]]
    Pmax, Pmin = gen[:, 8] / base, gen[:, 9] / base
    c1 = gcost[:, 5]
    f = [idx[int(b)] for b in br[:, 0]]; t = [idx[int(b)] for b in br[:, 1]]
    tap = np.where(br[:, 8] == 0, 1.0, br[:, 8])
    bsus = 1.0 / (br[:, 3] * tap)                           # branch susceptance 1/(x*tau)
    rate = br[:, 5] / base

    m = pyo.ConcreteModel()
    m.B, m.G, m.L = pyo.RangeSet(0, nb - 1), pyo.RangeSet(0, ng - 1), pyo.RangeSet(0, nl - 1)
    m.Pg = pyo.Var(m.G, bounds=lambda m, k: (Pmin[k], Pmax[k]), initialize=lambda m, k: (Pmin[k] + Pmax[k]) / 2)
    m.th = pyo.Var(m.B, bounds=(-np.pi, np.pi), initialize=0.0)
    m.obj = pyo.Objective(expr=base * sum(c1[k] * m.Pg[k] for k in m.G))

    flow = lambda m, l: bsus[l] * (m.th[f[l]] - m.th[t[l]])
    gens_at = {i: [k for k in range(ng) if gbus[k] == i] for i in range(nb)}
    from_at = {i: [l for l in range(nl) if f[l] == i] for i in range(nb)}
    to_at   = {i: [l for l in range(nl) if t[l] == i] for i in range(nb)}
    m.balance = pyo.Constraint(m.B, rule=lambda m, i: sum(m.Pg[k] for k in gens_at[i]) - Pd[i]
                               - sum(flow(m, l) for l in from_at[i]) + sum(flow(m, l) for l in to_at[i]) == 0)
    m.line = pyo.Constraint(m.L, rule=lambda m, l: (-rate[l], flow(m, l), rate[l]))
    m.ang = pyo.Constraint(m.L, rule=lambda m, l: (np.radians(br[l, 11]), m.th[f[l]] - m.th[t[l]], np.radians(br[l, 12])))
    m.refang = pyo.Constraint(expr=m.th[ref] == 0)
    t_build = time.perf_counter() - t0

    solver = pyo.SolverFactory("highs")
    t1 = time.perf_counter(); res = solver.solve(m); t_solve = time.perf_counter() - t1
    cond = res.solver.termination_condition
    return {"status": str(getattr(cond, "name", cond)), "cost": pyo.value(m.obj), "t_build": t_build, "t_solve": t_solve}


def solve_pypower_dc(case):
    c = {k: (v.copy() if hasattr(v, "copy") else v) for k, v in case.items()}; c["version"] = "2"
    r = dcopf(c, ppoption(PF_DC=1, VERBOSE=0, OUT_ALL=0))
    return {"success": bool(r["success"]), "cost": float(r["f"])}

## Sanity check on the 5-bus case
The PJM 5-bus data from `02_dcopf_pypower_reference.ipynb` should give the DC-OPF cost reported in `01_dcopf_pyomo.ipynb`, \$17,479.90/h.

In [6]:
pjm5 = {"baseMVA": 100.0,
  "bus": np.array([[1,2,0,0,0,0,1,1,0,230,1,1.1,.9],[2,1,300,98.61,0,0,1,1,0,230,1,1.1,.9],[3,2,300,98.61,0,0,1,1,0,230,1,1.1,.9],
                   [4,3,400,131.47,0,0,1,1,0,230,1,1.1,.9],[5,2,0,0,0,0,1,1,0,230,1,1.1,.9]], float),
  "gen": np.array([[1,20,0,30,-30,1,100,1,40,0],[1,85,0,127.5,-127.5,1,100,1,170,0],[3,260,0,390,-390,1,100,1,520,0],
                   [4,100,0,150,-150,1,100,1,200,0],[5,300,0,450,-450,1,100,1,600,0]], float),
  "gencost": np.array([[2,0,0,3,0,c,0] for c in (14,15,30,40,10)], float),
  "branch": np.array([[1,2,.00281,.0281,.00712,400,400,400,0,0,1,-30,30],[1,4,.00304,.0304,.00658,426,426,426,0,0,1,-30,30],
                      [1,5,.00064,.0064,.03126,426,426,426,0,0,1,-30,30],[2,3,.00108,.0108,.01852,426,426,426,0,0,1,-30,30],
                      [3,4,.00297,.0297,.00674,426,426,426,0,0,1,-30,30],[4,5,.00297,.0297,.00674,240,240,240,0,0,1,-30,30]], float)}
o5, p5 = solve_pyomo_dcopf(pjm5), solve_pypower_dc(pjm5)
print(f"5-bus  Pyomo/HiGHS ${o5['cost']:,.4f}/h | PYPOWER ${p5['cost']:,.4f}/h | notebook 01: $17,479.90/h")

5-bus  Pyomo/HiGHS $17,479.8969/h | PYPOWER $17,479.8969/h | notebook 01: $17,479.90/h


## Objective comparison across IEEE cases
Each case is solved once with each solver. Because DC-OPF is a linear program, both solvers should reach the same global optimum, so the differences should be at numerical-tolerance level (much smaller than in the AC case).

In [7]:
rows = []
for n, case in cases.items():
    o, p = solve_pyomo_dcopf(case), solve_pypower_dc(case)
    rows.append({"Case": n, "Buses": len(case["bus"]), "HiGHS status": o["status"], "PYPOWER converged": p["success"],
                 "Pyomo/HiGHS cost ($/h)": o["cost"], "PYPOWER cost ($/h)": p["cost"],
                 "Abs. diff ($/h)": abs(o["cost"] - p["cost"]), "Rel. diff": abs(o["cost"] - p["cost"]) / p["cost"]})
results = pd.DataFrame(rows)
results.style.format({"Pyomo/HiGHS cost ($/h)": "{:,.4f}", "PYPOWER cost ($/h)": "{:,.4f}",
                      "Abs. diff ($/h)": "{:.1e}", "Rel. diff": "{:.1e}"}).hide(axis="index")

Case,Buses,HiGHS status,PYPOWER converged,Pyomo/HiGHS cost ($/h),PYPOWER cost ($/h),Abs. diff ($/h),Rel. diff
14_ieee,14,optimal,True,"2,051.5263","2,051.5263",1.9e-08,9.0e-12
30_ieee,30,optimal,True,"7,504.4405","7,504.4405",1.8e-08,2.3e-12
39_epri,39,optimal,True,"136,816.1561","136,816.1561",1.3e-07,9.4e-13
57_ieee,57,optimal,True,"34,772.9479","34,772.9479",1.2e-07,3.4e-12
118_ieee,118,optimal,True,"93,132.6793","93,132.6793",6.8e-08,7.3e-13
162_ieee_dtc,162,optimal,True,"101,268.2940","101,268.2940",1.8e-07,1.8e-12


In [8]:
print(f"Largest relative difference over {len(results)} cases: {results['Rel. diff'].max():.1e}")
print(f"Largest absolute difference: ${results['Abs. diff ($/h)'].max():.1e}/h")
print("All converged:", bool((results['HiGHS status'] == 'optimal').all() and results['PYPOWER converged'].all()))

Largest relative difference over 6 cases: 9.0e-12
Largest absolute difference: $1.8e-07/h
All converged: True


## Solve time
Each case is run once as a warm-up (not counted) and then `N_RUNS` times; the table shows the **median** in milliseconds, which is robust to the occasional slow run. Times are wall-clock on the machine running the notebook, so absolute values will differ on yours.

* **Pyomo build**: constructing the Pyomo model.
* **HiGHS solve**: the `solver.solve(m)` call. It includes transferring the model to HiGHS and reading the solution back, not only the simplex or interior-point iterations.
* **Pyomo total**: build + solve.
* **PYPOWER total**: the whole `dcopf` call, including its own case conversion and matrix set-up, not only its PIPS interior-point solve.

The two totals are therefore comparisons of the two complete workflows, not of the bare solvers.

In [9]:
N_RUNS = 20
trows = []
for n, case in cases.items():
    solve_pyomo_dcopf(case); solve_pypower_dc(case)                       # warm-up
    build, solve, total, pp = [], [], [], []
    for _ in range(N_RUNS):
        o = solve_pyomo_dcopf(case); build.append(o["t_build"]); solve.append(o["t_solve"]); total.append(o["t_build"] + o["t_solve"])
        t0 = time.perf_counter(); solve_pypower_dc(case); pp.append(time.perf_counter() - t0)
    trows.append({"Case": n, "Buses": len(case["bus"]),
                  "Pyomo build (ms)": 1e3 * np.median(build), "HiGHS solve (ms)": 1e3 * np.median(solve),
                  "Pyomo total (ms)": 1e3 * np.median(total), "PYPOWER total (ms)": 1e3 * np.median(pp)})
timing = pd.DataFrame(trows)
timing.style.format({c: "{:.1f}" for c in timing.columns if "(ms)" in c}).hide(axis="index")

Case,Buses,Pyomo build (ms),HiGHS solve (ms),Pyomo total (ms),PYPOWER total (ms)
14_ieee,14,2.7,11.8,14.8,31.7
30_ieee,30,4.9,16.8,21.8,39.0
39_epri,39,5.3,19.5,25.1,74.8
57_ieee,57,7.4,25.3,32.7,44.0
118_ieee,118,16.6,50.0,68.0,99.0
162_ieee_dtc,162,29.6,66.1,95.2,211.0


In [10]:
print(f"Median of {N_RUNS} runs per case after one warm-up run.")
print(f"Slowest median HiGHS solve: {timing['HiGHS solve (ms)'].max():.1f} ms | slowest median Pyomo total: {timing['Pyomo total (ms)'].max():.1f} ms | slowest median PYPOWER total: {timing['PYPOWER total (ms)'].max():.1f} ms")

Median of 20 runs per case after one warm-up run.
Slowest median HiGHS solve: 66.1 ms | slowest median Pyomo total: 95.2 ms | slowest median PYPOWER total: 211.0 ms


**Reading the result.** The objectives agree to numerical tolerance on every case, as expected for an LP where both solvers find the global optimum. All cases solve in well under a second with either workflow, so for networks of this size the DC-OPF cost is dominated by set-up overhead rather than the optimisation itself. Timings on much larger networks, or relative speed on other machines, are not covered by this test.

In all six cases $G_s = 0$ (the shunts are purely susceptance), so the $G_s$ term, which matches PYPOWER's DC formulation, is not exercised by these cases.

Cases not included: IEEE 24 and 73 (quadratic generator costs, which would make the problem a QP) and IEEE 300 (phase-shifting transformers). Using them would require changing the objective or the branch model.